## Task 1

### Step 1

In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

products = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones", "Webcam"]
countries = ["USA", "Germany", "France", "UK", "Canada", "Japan"]
start_date = datetime(2024, 1, 1)

raw_orders = []

# 185 normal records 
for i in range(1, 186):
    order = {
        "order_id": i,
        "customer_id": random.randint(1000, 1100),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 10),
        "unit_price": round(random.uniform(10, 1500), 2),
        "order_date": (start_date + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
        "shipping_country": random.choice(countries)
    }
    raw_orders.append(order)

# Problematic records

problem_records = [

# Missing product_name
{"order_id": 186, "customer_id": 1005, "product_name": None, "quantity": 2, "unit_price": 50, "order_date": "2025-03-01", "shipping_country": "USA"},
{"order_id": 187, "customer_id": 1006, "product_name": "", "quantity": 3, "unit_price": 20, "order_date": "2025-02-10", "shipping_country": "UK"},
{"order_id": 188, "customer_id": 1007, "product_name": None, "quantity": 1, "unit_price": 15, "order_date": "2025-01-15", "shipping_country": "Germany"},

# Negative quantity or price
{"order_id": 189, "customer_id": 1008, "product_name": "Mouse", "quantity": -2, "unit_price": 25, "order_date": "2025-02-20", "shipping_country": "France"},
{"order_id": 190, "customer_id": 1009, "product_name": "Keyboard", "quantity": 2, "unit_price": -100, "order_date": "2025-02-21", "shipping_country": "Canada"},
{"order_id": 191, "customer_id": 1010, "product_name": "Monitor", "quantity": -1, "unit_price": 300, "order_date": "2025-02-22", "shipping_country": "USA"},

# Malformed dates
{"order_id": 192, "customer_id": 1011, "product_name": "Laptop", "quantity": 1, "unit_price": 900, "order_date": "not-a-date", "shipping_country": "Japan"},
{"order_id": 193, "customer_id": 1012, "product_name": "Mouse", "quantity": 4, "unit_price": 15, "order_date": "2025-13-45", "shipping_country": "Germany"},
{"order_id": 194, "customer_id": 1013, "product_name": "Keyboard", "quantity": 2, "unit_price": 45, "order_date": "2025/99/10", "shipping_country": "UK"},

# Duplicate order_id
{"order_id": 10, "customer_id": 1014, "product_name": "Monitor", "quantity": 1, "unit_price": 250, "order_date": "2025-01-01", "shipping_country": "USA"},
{"order_id": 20, "customer_id": 1015, "product_name": "Webcam", "quantity": 3, "unit_price": 80, "order_date": "2025-01-02", "shipping_country": "France"},
{"order_id": 30, "customer_id": 1016, "product_name": "Headphones", "quantity": 2, "unit_price": 120, "order_date": "2025-01-03", "shipping_country": "Canada"},

# Quantity or price as string
{"order_id": 195, "customer_id": 1017, "product_name": "Laptop", "quantity": "5", "unit_price": 950, "order_date": "2025-03-02", "shipping_country": "USA"},
{"order_id": 196, "customer_id": 1018, "product_name": "Mouse", "quantity": 2, "unit_price": "25", "order_date": "2025-03-03", "shipping_country": "Germany"},
{"order_id": 197, "customer_id": 1019, "product_name": "Keyboard", "quantity": "three", "unit_price": 40, "order_date": "2025-03-04", "shipping_country": "UK"},
]

raw_orders.extend(problem_records)

#  Fill remaining to reach 200
while len(raw_orders) < 200:
    i = len(raw_orders) + 1
    raw_orders.append({
        "order_id": i,
        "customer_id": random.randint(1000, 1100),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 10),
        "unit_price": round(random.uniform(10, 1500), 2),
        "order_date": (start_date + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
        "shipping_country": random.choice(countries)
    })

print(len(raw_orders))

200


### Step 2

In [2]:
from datetime import datetime

def extract(raw_records):

    required_fields = [
        "order_id",
        "customer_id",
        "product_name",
        "quantity",
        "unit_price",
        "order_date",
        "shipping_country"
    ]

    valid_records = []
    rejected_records = []

    for record in raw_records:
        try:
            # required fields check
            if any(field not in record or record[field] in [None, ""] for field in required_fields):
                rejected_records.append({
                    "record": record,
                    "reason": "Missing required field"
                })
                continue

            # quantity check
            if not isinstance(record["quantity"], (int, float)) or record["quantity"] <= 0:
                rejected_records.append({
                    "record": record,
                    "reason": "Invalid quantity"
                })
                continue

            # price check
            if not isinstance(record["unit_price"], (int, float)) or record["unit_price"] <= 0:
                rejected_records.append({
                    "record": record,
                    "reason": "Invalid unit_price"
                })
                continue

            # date check
            try:
                datetime.strptime(record["order_date"], "%Y-%m-%d")
            except:
                rejected_records.append({
                    "record": record,
                    "reason": "Invalid order_date"
                })
                continue

            valid_records.append(record)

        except:
            rejected_records.append({
                "record": record,
                "reason": "Unexpected error"
            })

    return valid_records, rejected_records


valid, rejected = extract(raw_orders)

print("Valid:", len(valid))
print("Rejected:", len(rejected))

Valid: 188
Rejected: 12


### Step 3

In [3]:
import pandas as pd
import numpy as np

def transform(valid_records):
    cleaned_df = pd.DataFrame(valid_records)
    
    # total amount
    cleaned_df["total_amount"] = cleaned_df["quantity"] * cleaned_df["unit_price"]
    
    # convert order_date to datetime
    cleaned_df["order_date"] = pd.to_datetime(cleaned_df["order_date"])
    
    # extract month and day of week
    cleaned_df["order_month"] = cleaned_df["order_date"].dt.month
    cleaned_df["order_day_of_week"] = cleaned_df["order_date"].dt.dayofweek  # Monday=0
    
    # format shipping_country
    cleaned_df["shipping_country"] = cleaned_df["shipping_country"].str.title()
    
    # remove duplicate order_id
    cleaned_df = cleaned_df.drop_duplicates(subset="order_id", keep="first")
    
    # ensure unit_price is numeric
    cleaned_df["unit_price"] = pd.to_numeric(cleaned_df["unit_price"], errors="coerce")
    
    # categorize amount based on unit_price
    conditions = [
        cleaned_df["unit_price"] < 25,
        (cleaned_df["unit_price"] >= 25) & (cleaned_df["unit_price"] < 100),
        cleaned_df["unit_price"] >= 100
    ]
    choices = ["small", "medium", "large"]
    
    cleaned_df["amount_category"] = np.select(conditions, choices, default="unknown")
    
    return cleaned_df

# usage
cleaned_df = transform(valid)
cleaned_df.head()


,order_id,customer_id,product_name,quantity,unit_price,order_date,shipping_country,total_amount,order_month,order_day_of_week,amount_category
0,1,1091,Monitor,1,507.09,2024-03-24,Uk,507.09,3,6,large
1,2,1014,Monitor,10,737.58,2024-07-31,Germany,7375.80,7,2,large
2,3,1088,Headphones,5,472.61,2024-10-02,Japan,2363.05,10,2,large
3,4,1038,Webcam,10,935.90,2024-10-07,Canada,9359.00,10,0,large
4,5,1035,Laptop,1,109.68,2024-06-15,Japan,109.68,6,5,large


### Step 4

In [4]:
def load(df, path):
    df.to_parquet(path, index=False)
   
    df_back = pd.read_parquet(path)
    return df_back


loaded_df = load(cleaned_df, "data.parquet")

row_match = len(cleaned_df) == len(loaded_df)
dtype_match = (cleaned_df.dtypes == loaded_df.dtypes).all()


print(f"Row count match: {'Yes' if row_match else 'No'} ({len(loaded_df)} rows)")
print(f"Dtypes match: {'Yes' if dtype_match else 'No'}")



Row count match: Yes (185 rows)
Dtypes match: Yes


### Step 5

In [5]:
valid, rejected = extract(raw_orders)
cleaned_df = transform(valid)
loaded_df = load(cleaned_df, "data.parquet")


summary_stats = {
    "Raw Records": len(raw_orders),
    "Valid Records": len(valid),
    "Rejected Records": len(rejected),
    "Transformed Records": len(cleaned_df),
    "Loaded Records": len(loaded_df)
}

summary_stats

{'Raw Records': 200,
 'Valid Records': 188,
 'Rejected Records': 12,
 'Transformed Records': 185,
 'Loaded Records': 185}

## Task 2

### Step 1

In [6]:
def extract_minimal(raw_data):
    valid_records = []
    rejected_records = []

    required_keys = {"order_id", "customer_id", "product_name", "quantity", "unit_price", "order_date","shipping_country"}
    
    for record in raw_data:
        if not isinstance(record, dict):
            rejected_records.append({"data": record, "reason": "Not a dictionary"})
            continue
            
        if required_keys.issubset(record.keys()):
            valid_records.append(record)
        else:
            rejected_records.append({"data": record, "reason": "Missing required keys"})
            
    return valid_records, rejected_records

valid, rejected = extract_minimal(raw_orders)

print(len(valid))
print(len(rejected))

200
0


In [7]:
def load_raw(raw_data, path):
    df_raw = pd.DataFrame(raw_data)

    
    df_raw['quantity'] = pd.to_numeric(df_raw['quantity'], errors='coerce')
    df_raw['unit_price'] = pd.to_numeric(df_raw['unit_price'], errors='coerce')
    
    df_raw.to_parquet(path, index=False)
    print(f"Raw data was loaded: {path} ({len(df_raw)} records)")
    return path

file_path = load_raw(valid, "data_lake.parquet")

Raw data was loaded: data_lake.parquet (200 records)


In [34]:
import pandas as pd
import numpy as np

def transform_raw(path):
    df = pd.read_parquet(path)
    
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    
    df["total_amount"] = df["quantity"] * df["unit_price"]
    
    df["order_month"] = df["order_date"].dt.month
    df["order_day_of_week"] = df["order_date"].dt.dayofweek
    
    df["shipping_country"] = df["shipping_country"].str.title()
    
    df = df.drop_duplicates(subset="order_id", keep="first")
    
    conditions = [
        df["unit_price"] < 25,
        (df["unit_price"] >= 25) & (df["unit_price"] < 100),
        df["unit_price"] >= 100
    ]
    choices = ["small", "medium", "large"]
    df["amount_category"] = np.select(conditions, choices, default="unknown")
    
    df = df.dropna(subset=["total_amount", "order_date"])
    
    return df

cleaned_df = transform_raw("data_lake.parquet")

print(len(cleaned_df))

193


### Step 2

#### How many records made it to the destination in each approach?
#### ETL : 185 records
#### ELT : 193 records

#### At what stage were problems caught in each?
#### ETL : before loading - at tranform stage
#### ELT : after loading - at transform stage

#### What are the advantages of each approach?
#### In ETL approach, an advantage is storage efficinecy, it means we only store clean, high-quality data. No space is wasted for invalid records. Another advantage is being data structure ready for analysis
#### in ELT approach, an advantage is speed, it means we don't wait for cleaning process for saving data to Data Lake. another advantage is that this approach preserves raw data  becuase raw data can be needed later

#### When would you choose one over the other?
#### I would choose ETL approach when I work with small data and for getting high secutity
#### I would choose ELT approach when working with big data and for preserving raw data

## Task 3

### Step 1

In [11]:
# Shared database (simulated)
database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():
    temp_data = {} 

    
    for order in database["orders"]:
        cid = order["customer_id"]
        if cid not in temp_data:
            temp_data[cid] = {"prices": [], "dates": []}
        
        temp_data[cid]["prices"].append(order["price"])
        temp_data[cid]["dates"].append(order["date"])
                                                                                                                                                 
    for customer_id, data in temp_data.items():
        total_orders = len(data["prices"])
        avg_amount = sum(data["prices"]) / total_orders
        last_order_date = max(data["dates"])
        
       
        database["features"][customer_id] = {
            "total_orders": total_orders,
            "average_amount": avg_amount,
            "last_order_date": last_order_date
        }

def prediction_service_read(customer_id):
    
    customer_features = database["features"].get(customer_id)

    if not customer_features:
        return f"Customer {customer_id}: No features calculated yet (New User)."


    avg = customer_features["average_amount"]
    total = customer_features["total_orders"]

    if avg > 150 and total > 2:
        prediction = "High Propensity to Buy (VIP)"
    elif avg > 50:
        prediction = "Medium Propensity"
    else:
        prediction = "Low Propensity"

    return f"Prediction for {customer_id}: {prediction} (Avg: {avg}, Total: {total})"

In [13]:
order_service_write({"customer_id": "C1", "price": 200, "date": "2024-03-01"})
order_service_write({"customer_id": "C1", "price": 300, "date": "2024-03-05"})
feature_service_compute()
print(prediction_service_read("C1"))

### Step 2

In [25]:
def order_service_get_raw_data(c_id):
    return [order for order in database["orders"] if order["customer_id"] == c_id]


def feature_service_get_features(c_id):

   raw_orders = order_service_get_raw_data(c_id)
    
   if not raw_orders:
        return None
        
    
   prices = [o["price"] for o in raw_orders]
   dates = [o["date"] for o in raw_orders]
    
   return {
        "avg_amount": sum(prices) / len(prices),
        "total_orders": len(prices),
        "last_order_date": max(dates)}


def prediction_service_read(c_id):
    
    features = feature_service_get_features(c_id)
    

    if not features: return "User not found"
    return "VIP" if features["avg_amount"] > 100 else "Regular"    
    
    

### Step 3

In [28]:
class MessageBroker:
    def __init__(self):
        self.subscriptions = {}
        self.latest_messages = {}

    def subscribe(self, topic, callback):
        if topic not in self.subscriptions:
            self.subscriptions[topic] = []
        self.subscriptions[topic].append(callback)

    def publish(self, topic, message):
        self.latest_messages[topic] = message
        
        if topic in self.subscriptions:
            for func in self.subscriptions[topic]:
                func(message)

broker = MessageBroker()
feature_db = {}    

In [29]:
def feature_service_on_order(order):
    c_id = order["customer_id"]
    price = order["price"]
    
  
    if c_id not in feature_db:
        feature_db[c_id] = []
    feature_db[c_id].append(price)
    
    avg = sum(feature_db[c_id]) / len(feature_db[c_id])
    
    
    features = {"customer_id": c_id, "avg_amount": avg}
    broker.publish("features", features)
    print(f"Feature Service: {c_id} üçün yeni ortalama {avg} hesablandı və paylaşıldı.")

broker.subscribe("orders", feature_service_on_order)

In [30]:
def prediction_service_on_features(features):
    c_id = features["customer_id"]
    avg = features["avg_amount"]
    
    status = "VIP" if avg > 100 else "Standard"
    print(f"Prediction Service: {c_id} statusu müəyyən edildi -> {status}")


broker.subscribe("features", prediction_service_on_features)

### Step 4

In [32]:
test_orders = [
    {"customer_id": "C1", "price": i * 10, "date": f"2026-03-{i+1:02d}"} 
    for i in range(1, 21)
]


db_shared = {"orders": [], "features": {}}
for o in test_orders: db_shared["orders"].append(o)

def batch_compute():
    prices = [o["price"] for o in db_shared["orders"]]
    db_shared["features"]["C1"] = {"avg": sum(prices)/len(prices)}
batch_compute()
res_shared = db_shared["features"]["C1"]["avg"]


def api_order_service(): return test_orders
def api_feature_service():
    orders = api_order_service()
    prices = [o["price"] for o in orders]
    return {"avg": sum(prices)/len(prices)}
res_api = api_feature_service()["avg"]


broker = MessageBroker()
pb_history = []
res_pb = {"avg": 0}

def pb_on_order(order):
    pb_history.append(order["price"])
    avg = sum(pb_history) / len(pb_history)
    broker.publish("features", {"avg": avg})

def pb_on_feature(feat):
    res_pb["avg"] = feat["avg"]

broker.subscribe("orders", pb_on_order)
broker.subscribe("features", pb_on_feature)

for o in test_orders:
    broker.publish("orders", o)


print(f"Shared DB Avg: {res_shared}")
print(f"API Avg:      {res_api}")
print(f"Pub/Sub Avg:  {res_pb['avg']}")
print(f"Matching:     {res_shared == res_api == res_pb['avg']}")

Shared DB Avg: 105.0
API Avg:      105.0
Pub/Sub Avg:  105.0
Matching:     True


#### Coupling: Tight. Every service is locked into the same central database schema; changing the table structure requires updating all services.

#### Latency: High. Updates only occur during scheduled "batch" runs, meaning the prediction service often uses stale data until the next cycle.

#### If the processing service fails, the system stays online but features remain frozen in time until the service is restored.

## Task 4

### Step 1

In [33]:
def run_batch_feature_update():
   
    customer_ids = set(o["customer_id"] for o in database["orders"])
    
    for c_id in customer_ids:
      
        customer_orders = [o["price"] for o in database["orders"] if o["customer_id"] == c_id]
        
        total_orders_per_day

        order_count_per_day = {}
        
        for order in database["orders"]:
            c_id = order["customer_id"]
            
            order_count_per_day[c_id] = order_count_per_day.get(c_id, 0) + 1
            
        
